# Energy Production and Consumption

The data in this notebook is taken from [this Kaggle Dataset](https://www.kaggle.com/datasets/stefancomanita/hourly-electricity-consumption-and-production/data), which has hourly electricity statistics for the Romanian grid over six consecutive years.

We use three samples:

- `sampled.csv` — 2000 data points evenly spaced over the six-year period,
- `first60.csv` — the first 60 days,
- `last60.csv` — the last 60 days.

The specifications live in [`Energy.lilo`](./Energy.lilo) (with shared units and helpers in [`Energy.lilo`](./Energy.lilo)). This notebook drives the SpecForge server to **check satisfiability**, **monitor** the specs against the data, **exemplify** a spec (synthesise data that satisfies it), and **export** a spec to RTAMT.

## Setup

First, import the SDK and connect to the SpecForge server.

In [ ]:
# Import the SpecForge SDK and necessary libraries
from specforge_sdk import (
    SpecForgeClient,
    nested_encoding,
    flat_encoding,
    EXPORT_LILO,
    EXPORT_JSON,
    EXPORT_RTAMT,
)
import pandas as pd
import json
import numpy as np

import os

In [ ]:
try:
    # Attempt to discover the port automatically
    specforgeClient = SpecForgeClient()
except RuntimeError:
    # If automatic discovery fails,
    #   use the environment variable SPECFORGE_PORT
    #   or if it is not set, default to 8080
    print("⚠️ Could not discover the SpecForge server port automatically.")
    port = os.environ.get("SPECFORGE_PORT", "8080")
    specforgeClient = SpecForgeClient(base_url="http://localhost:" + port)

# Call the Health Check endpoint to verify the connection
if specforgeClient.health_check():
    print(f"✓ Connected to SpecForge API v{specforgeClient.version()}")
else:
    print("✗ Cannot connect to SpecForge API")
    print("Make sure the SpecForge server is running on http://localhost:" + port)

## Satisfiability Checks

Before monitoring concrete data, we can ask whether an individual spec, or the whole specification system together with its assumptions, is even satisfiable.

In [ ]:
specforgeClient.check_satisfiability(
    system="Energy", definition="noSolarAtNight", return_details=True
)

In [ ]:
# The whole system: all specs plus the (hard) assumptions, checked together.
specforgeClient.check_satisfiability(system="Energy")

You can use the `.search()` method to find specs using a query. For example, the query `label:consumption` will return all specs with the label `consumption`. Refer to the [spec search page](https://docs.imiron.io/v/latest/en/spec-search.html) in the manual for details. The results are grouped by the system the specs belong to.

In [ ]:
consumptionSpecs = specforgeClient.search("label:consumption")
print(consumptionSpecs)

This list of specs can be provided to `.check_satisfiability(definition=...)` to check whether the specs are jointly satisfiable.

In [ ]:
specforgeClient.check_satisfiability(
    system="Energy", definition=consumptionSpecs["Energy"]
)

## Importing Data

The three data files correspond to the first 60 days, the last 60 days, and a sampled version of 2000 evenly spaced entries from the entire dataset. Their columns line up with the signals declared in `Energy.lilo`.

In [ ]:
first60 = pd.read_csv("first60.csv")
last60 = pd.read_csv("last60.csv")
sampled = pd.read_csv("sampled.csv")

In [ ]:
first60.head()

In [ ]:
last60.head()

In [ ]:
sampled.head()

## Exploration — time-of-day behaviour

We start with specs keyed off the time of day:

- there is no solar generation at night,
- during the day, solar soon overtakes wind,
- peak hours run above a multiple of average consumption,
- at night the grid keeps a surplus.

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="noSolarAtNight", data_file="first60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="daytimeSolarDominance", data_file="first60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="peakDemandPeriod", data_file="first60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="surplusAtNight", data_file="first60.csv"
)

## The renewable mix

Next we look at the balance between renewable, low-carbon and fossil sources:

- at night, wind dominance eventually sustains for a couple of hours,
- renewables recur as the majority source within every twelve-hour window,
- the mix is genuinely diverse (every clean source contributes),
- the source leading the mix is often a low-carbon one.

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="nighttimeWindDominance", data_file="last60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="frequentlyMoreRenewable", data_file="last60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="diverseMix", data_file="last60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="lowCarbonDominance", data_file="last60.csv"
)

## Units, sliding windows and ramps

These specs exercise the more quantitative features — units of measure (`MW`, `MWh`), the `max`/`min` built-ins, sliding windows, and step operators:

- a full day at the current production level clears the baseload energy need,
- the gap between supply and demand stays bounded,
- the hour-to-hour production ramp stays within a slew limit.

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="dailyEnergyMeetsBaseload", data_file="last60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="spreadBounded", data_file="last60.csv"
)

In [ ]:
specforgeClient.monitor(
    system="Energy", definition="rampLimited", data_file="last60.csv"
)

## Exemplification

Rather than checking a spec against *existing* data, **exemplification** asks the solver to *synthesise* a short trace that satisfies it. This is useful for understanding what a spec actually permits.

The generated trace must also respect the system's **hard** assumptions (`productionNonNegative`, `renewableBounded`), while the **soft** assumption (`consumptionPositive`, tagged `#[rigidity = "soft"]`) is treated with lower priority. With `fix_defaults=True`, any `param` with a `#[default = ...]` is pinned to that default — here `surplusMargin = 300<MW>` and `peakFactor = 1.2`.

In [ ]:
# Synthesise data satisfying `surplusHealthy`, with params fixed to their defaults.
specforgeClient.exemplify(
    system="Energy",
    definition="surplusHealthy",
    n_points=5,
    fix_defaults=True,
)

In [ ]:
# We can add extra assumptions to steer the example. Each assumption is a Lilo
# expression with a rigidity ("Hard" or "Soft"). Here we force the interesting
# branch of `peakDemandPeriod` by assuming `peakHours` holds.
specforgeClient.exemplify(
    system="Energy",
    definition="peakDemandPeriod",
    n_points=6,
    assumptions=[{"expression": "peakHours", "rigidity": "Hard"}],
    fix_defaults=True,
)

## Exporting to RTAMT

SpecForge can export a spec to [RTAMT](https://github.com/nickovic/rtamt) (an STL monitoring library). The RTAMT grammar is narrower than Lilo's, so only an **export-clean** subset works: no strings, records, `if`/`then`/`else`, the `time` built-in, or the change operators (`will_change` / `did_change` / `*_with`).

`daytimeSolarDominance` is purely temporal arithmetic over numeric signals, so it exports cleanly:

In [ ]:
print(
    specforgeClient.export(
        system="Energy",
        definition="daytimeSolarDominance",
        export_type=EXPORT_RTAMT,
        return_string=True,
    )
)

By contrast, `weekdayBoundaryAhead` uses the `will_change` change operator (and the string-valued `weekday` signal), which RTAMT cannot represent — so the export is rejected:

In [ ]:
try:
    specforgeClient.export(
        system="Energy",
        definition="weekdayBoundaryAhead",
        export_type=EXPORT_RTAMT,
        return_string=True,
    )
except Exception as e:
    print("Export rejected, as expected:")
    print(e)